# for lolz

In [ ]:
from pathlib import Path
import json

import numpy as np

from util.flatten_discord_export import flatten_export
from util.hierarchical_cluster_message_bodies import (
    DEFAULT_CACHE,
    DEFAULT_PROJECTION,
    build_cluster_labels,
    embed_records,
    fit_hdbscan,
    load_body_records,
    pca_clustering_space,
    write_projection_json,
    write_report,
)

In [ ]:
input_path = Path("data/sample-data.json")
flat_output_path = Path("data/sample-data.flat.json")
hierarchy_output_path = Path("reports/message_body_hierarchy.md")
projection_output_path = DEFAULT_PROJECTION
embedding_cache_path = DEFAULT_CACHE

model = "qwen3-embedding"
batch_size = 16
random_state = 7

## Flatten Export

In [ ]:
export = json.loads(input_path.read_text())
flattened = flatten_export(export)

flat_output_path.parent.mkdir(parents=True, exist_ok=True)
flat_output_path.write_text(json.dumps(flattened, indent=2, ensure_ascii=False) + "\n")

len(flattened), flattened[:5]

## Hierarchical Message Body Clusters

In [ ]:
records, buckets = load_body_records(input_path, filter_short_bodies=True)
hierarchy_embeddings = embed_records(
    records=records,
    model=model,
    batch_size=batch_size,
    cache_path=embedding_cache_path,
)
vectors, clustering_note = pca_clustering_space(
    hierarchy_embeddings,
    dimensions=50,
    random_state=random_state,
)
labels = fit_hdbscan(vectors, min_cluster_size=12, min_samples=2)
cluster_labels = build_cluster_labels(
    records=records,
    vectors=vectors,
    labels=labels,
    samples=12,
    min_subcluster_size=8,
    min_samples=2,
    max_subclusters=8,
)

write_report(
    output=hierarchy_output_path,
    records=records,
    buckets=buckets,
    vectors=vectors,
    top_labels=labels,
    min_subcluster_size=8,
    min_samples=2,
    samples=12,
    max_subclusters=8,
    clustering_note=clustering_note,
    cluster_labels=cluster_labels,
)
write_projection_json(
    output=projection_output_path,
    records=records,
    vectors=vectors,
    top_labels=labels,
    min_subcluster_size=8,
    min_samples=2,
    max_subclusters=8,
    random_state=random_state,
    clustering_note=clustering_note,
    cluster_labels=cluster_labels,
)

clusters = len({label for label in labels if label != -1})
noise = int(np.sum(labels == -1))
held_out = sum(len(values) for values in buckets.values())

{
    "report": str(hierarchy_output_path),
    "projection": str(projection_output_path),
    "clusters": clusters,
    "outliers": noise,
    "held_out": held_out,
    "clustering_space": clustering_note,
}